# Feature Engineering 🔧

"Garbage in, garbage out." Features are everything in machine learning. Better features → better models.

## What You'll Learn
- Feature types and representation
- Handling missing data
- Scaling and normalization
- Encoding categorical variables
- Feature creation (polynomial, interactions)
- Feature selection
- Real-world data cleaning

## Why Feature Engineering Matters 💎

### The Impact

```
Raw Data Features:  Model scores 75% accuracy
         ↓
Feature Engineering
         ↓
Good Features:      Model scores 92% accuracy
         ↓
Advanced Algorithm  (XGBoost vs Random Forest)
         ↓
Final Model:        Model scores 93% accuracy
```

**Feature engineering: +17%**
**Algorithm choice: +1%**

### Quote from Kaggle Masters
"Feature engineering beats fancy algorithms."

Spending 80% of time on features, 20% on algorithms is realistic!

## Feature Types 📊

### Numerical Features
- **Continuous:** Any number (height: 5.8, price: $199.99)
- **Discrete:** Integer counts (age: 25, number of items: 3)

### Categorical Features
- **Nominal:** No order (color: red/blue/green, country: USA/UK/CA)
- **Ordinal:** Has order (rating: poor/fair/good/excellent, education: high school/bachelor/masters)

### Temporal Features
- **Date/Time:** Timestamps, can extract day/month/hour/etc

### Text Features
- **Unstructured:** Raw text (reviews, tweets)
- Requires special handling

## Handling Missing Data 🔍

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error

print("📚 Libraries loaded!")

In [ ]:
# Create dataset with missing values
data = {
    'Age': [25, 30, np.nan, 35, 40, np.nan, 28, 32, 29, 31],
    'Income': [50000, 60000, 55000, np.nan, 75000, 65000, np.nan, 70000, 58000, 62000],
    'Experience': [1, 5, 3, 10, 15, 8, 4, 9, 2, 6],
    'Purchased': [0, 1, 0, 1, 1, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
print("Original Dataset with Missing Values:")
print(df)
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Strategy 1: Remove rows with missing values
df_dropped = df.dropna()
print("\n" + "="*60)
print("STRATEGY 1: DROP ROWS WITH MISSING DATA")
print("="*60)
print(f"Original rows: {len(df)}")
print(f"After dropping: {len(df_dropped)}")
print(f"Data lost: {len(df) - len(df_dropped)} rows ({(len(df) - len(df_dropped))/len(df)*100:.0f}%)")
print("\n❌ Problem: We lost 30% of data!")
print("✅ Good for: Very small amount of missing data (< 5%)")

In [ ]:
# Strategy 2: Imputation - fill with mean
df_imputed_mean = df.copy()
imputer_mean = SimpleImputer(strategy='mean')
df_imputed_mean[['Age', 'Income']] = imputer_mean.fit_transform(df[['Age', 'Income']])

print("\n" + "="*60)
print("STRATEGY 2: IMPUTE WITH MEAN")
print("="*60)
print(df_imputed_mean)
print("\n✅ Pros: Keep all data, simple")
print("❌ Cons: Ignores relationships, reduces variance")

In [ ]:
# Strategy 3: Forward/Backward fill (for time-series)
df_ffill = df.copy()
df_ffill['Age'] = df_ffill['Age'].fillna(method='ffill')  # Forward fill
df_ffill['Income'] = df_ffill['Income'].fillna(method='bfill')  # Backward fill

print("\n" + "="*60)
print("STRATEGY 3: FORWARD/BACKWARD FILL (Time-Series)")
print("="*60)
print(df_ffill)
print("\n✅ Good for: Sequential data (time-series)")
print("❌ Not good for: Random missing data")

In [ ]:
# Strategy 4: Advanced - KNN imputation
from sklearn.impute import KNNImputer

df_knn = df.copy()
imputer_knn = KNNImputer(n_neighbors=3)
df_knn[['Age', 'Income']] = imputer_knn.fit_transform(df[['Age', 'Income']])

print("\n" + "="*60)
print("STRATEGY 4: KNN IMPUTATION (ADVANCED)")
print("="*60)
print(df_knn)
print("\n✅ Pros: Uses feature relationships, more sophisticated")
print("❌ Cons: Slower, needs to choose k")
print("💡 Best for: When features are related")

In [ ]:
# Strategy 5: Create indicator (mark which were imputed)
df_indicator = df.copy()
df_indicator['Age_Missing'] = df_indicator['Age'].isnull().astype(int)
df_indicator['Income_Missing'] = df_indicator['Income'].isnull().astype(int)

# Then impute
imputer = SimpleImputer(strategy='mean')
df_indicator[['Age', 'Income']] = imputer.fit_transform(df_indicator[['Age', 'Income']])

print("\n" + "="*60)
print("STRATEGY 5: INDICATOR + IMPUTATION")
print("="*60)
print(df_indicator)
print("\n✅ Pros: Model learns which values were imputed")
print("💡 Often best for: Machine learning (model uses this info)")

## Scaling & Normalization 📏

In [ ]:
# Create data with different scales
data_scaling = {
    'Age': [20, 25, 30, 35, 40],
    'Income': [30000, 50000, 70000, 90000, 110000],
    'Hours_Worked': [35, 40, 40, 45, 50]
}

df_scaling = pd.DataFrame(data_scaling)
print("Raw Data (Different Scales):")
print(df_scaling)
print(f"\nStatistics:")
print(df_scaling.describe())

In [ ]:
# Method 1: Standardization (Z-score normalization)
scaler_std = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_std.fit_transform(df_scaling),
    columns=df_scaling.columns
)

print("\n" + "="*60)
print("METHOD 1: STANDARDIZATION (Z-score)")
print("="*60)
print("Formula: (x - mean) / std")
print("\nResult:")
print(df_standardized)
print(f"\nMean: {df_standardized.mean().values}")
print(f"Std:  {df_standardized.std().values}")
print("\n✅ Mean ≈ 0, Std ≈ 1")
print("✅ Good for: Neural networks, SVM, KNN")

In [ ]:
# Method 2: Min-Max Scaling (Normalization)
scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(df_scaling),
    columns=df_scaling.columns
)

print("\n" + "="*60)
print("METHOD 2: MIN-MAX SCALING (Normalization)")
print("="*60)
print("Formula: (x - min) / (max - min)")
print("\nResult:")
print(df_normalized)
print(f"\nMin: {df_normalized.min().values}")
print(f"Max: {df_normalized.max().values}")
print("\n✅ All values between 0 and 1")
print("✅ Good for: Trees (robust to scale), image data")

In [ ]:
# Visualize the effect
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original
for col in df_scaling.columns:
    axes[0].scatter(range(len(df_scaling)), df_scaling[col], label=col, s=100, alpha=0.7)
axes[0].set_title('Original Data\n(Different Scales)', fontsize=12)
axes[0].set_ylabel('Value', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Standardized
for col in df_standardized.columns:
    axes[1].scatter(range(len(df_standardized)), df_standardized[col], label=col, s=100, alpha=0.7)
axes[1].set_title('Standardized\n(Mean=0, Std=1)', fontsize=12)
axes[1].set_ylabel('Value', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Normalized
for col in df_normalized.columns:
    axes[2].scatter(range(len(df_normalized)), df_normalized[col], label=col, s=100, alpha=0.7)
axes[2].set_title('Normalized\n(Min=0, Max=1)', fontsize=12)
axes[2].set_ylabel('Value', fontsize=11)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice how all methods put features on comparable scales!")

In [ ]:
# When to scale?
print("\n" + "="*60)
print("WHEN TO SCALE?")
print("="*60)

scaling_guide = """
SCALE/NORMALIZE:
  ✅ Neural Networks (always!)
  ✅ SVM, KNN, Logistic Regression
  ✅ Any distance-based algorithm
  ✅ When features have very different ranges

DON'T SCALE:
  ❌ Decision Trees, Random Forests, XGBoost
     (trees are scale-invariant)
  ❌ If all features already similar scale

BEST PRACTICE:
  1. Always scale for neural networks
  2. Fit scaler on training data only
  3. Apply same scaler to test data
  4. Don't scale tree-based models (waste of time)
"""

print(scaling_guide)

## Encoding Categorical Variables 🏷️

In [ ]:
# Dataset with categorical features
data_cat = {
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Blue'],
    'Size': ['Small', 'Large', 'Medium', 'Small', 'Large'],
    'Price': [100, 200, 150, 110, 210]
}

df_cat = pd.DataFrame(data_cat)
print("Dataset with Categorical Features:")
print(df_cat)
print(f"\nData types:")
print(df_cat.dtypes)
print("\n⚠️  Problem: Most ML models need numbers, not text!")

In [ ]:
# Method 1: Label Encoding (Ordinal)
df_label = df_cat.copy()

le_color = LabelEncoder()
le_size = LabelEncoder()

df_label['Color'] = le_color.fit_transform(df_label['Color'])
df_label['Size'] = le_size.fit_transform(df_label['Size'])

print("\n" + "="*60)
print("METHOD 1: LABEL ENCODING")
print("="*60)
print(df_label)

print(f"\nColor mapping: {dict(zip(le_color.classes_, le_color.transform(le_color.classes_)))}")
print(f"Size mapping: {dict(zip(le_size.classes_, le_size.transform(le_size.classes_)))}")

print("\n⚠️  Problem: Model might think Large (2) > Medium (0) > Small (1)")
print("           (Creates false ordering!)")
print("❌ Not good for: Nominal categories (no order)")
print("✅ Good for: Ordinal categories (has order)")

In [ ]:
# Method 2: One-Hot Encoding (Categorical)
df_onehot = pd.get_dummies(df_cat, columns=['Color', 'Size'], drop_first=False)

print("\n" + "="*60)
print("METHOD 2: ONE-HOT ENCODING")
print("="*60)
print(df_onehot)

print("\n✅ Each category gets its own column (0 or 1)")
print("✅ No false ordering created")
print("❌ Creates more columns (problem with many categories)")
print("✅ Good for: Nominal categories (no order)")
print("⚠️  Note: Can drop one column per feature to avoid collinearity")

In [ ]:
# One-hot with drop_first=True (prevents multicollinearity)
df_onehot_drop = pd.get_dummies(df_cat, columns=['Color', 'Size'], drop_first=True)

print("\n" + "="*60)
print("METHOD 2B: ONE-HOT ENCODING (DROP FIRST)")
print("="*60)
print(df_onehot_drop)

print("\n💡 If Color_Red=0 and Color_Blue=0, we know it's Green")
print("   No need for Color_Green column (reduces collinearity)")
print("✅ Best practice: drop_first=True")

In [ ]:
# Decision guide
print("\n" + "="*60)
print("ENCODING DECISION GUIDE")
print("="*60)

guide = """
┌──────────────────────┬─────────────┬──────────────────┐
│ Encoding Method      │ Use When    │ Best For         │
├──────────────────────┼─────────────┼──────────────────┤
│ Label Encoding       │ Ordinal     │ Ordered          │
│ (0, 1, 2, ...)       │ categories  │ categories       │
│                      │             │ (Low/Med/High)   │
├──────────────────────┼─────────────┼──────────────────┤
│ One-Hot Encoding     │ Nominal     │ Unordered cats   │
│ (0/1 columns)        │ categories  │ (Red/Blue/Green) │
│                      │             │ < 10 categories  │
├──────────────────────┼─────────────┼──────────────────┤
│ Target Encoding      │ High-card   │ Many categories  │
│ (mean target value)  │ nominal     │ (City: 1000s)    │
│                      │             │                  │
├──────────────────────┼─────────────┼──────────────────┤
│ Binary Encoding      │ High-card   │ Tree-based       │
│ (binary representation)│ nominal     │ (XGBoost)        │
└──────────────────────┴─────────────┴──────────────────┘

QUICK RULE:
  • Few categories & no order → One-Hot Encoding
  • Few categories & has order → Label Encoding
  • Many categories → Target or Binary Encoding
"""

print(guide)

## Creating New Features 🎨

In [ ]:
# Original features
data_feat = {
    'Height_cm': [160, 170, 180, 165, 175],
    'Weight_kg': [60, 75, 85, 65, 80],
    'Age': [20, 25, 30, 22, 28]
}

df_feat = pd.DataFrame(data_feat)
print("Original Features:")
print(df_feat)

In [ ]:
# Feature engineering: Create new meaningful features

# 1. Polynomial features
df_feat['Height_squared'] = df_feat['Height_cm'] ** 2

# 2. Interaction features (feature combinations)
df_feat['Height_x_Weight'] = df_feat['Height_cm'] * df_feat['Weight_kg']

# 3. Domain-specific features (BMI)
df_feat['BMI'] = df_feat['Weight_kg'] / ((df_feat['Height_cm'] / 100) ** 2)

# 4. Ratio features
df_feat['Weight_per_cm'] = df_feat['Weight_kg'] / df_feat['Height_cm']

# 5. Binning/Discretization
df_feat['Age_Group'] = pd.cut(df_feat['Age'], bins=[0, 25, 30, 100], 
                               labels=['Young', 'Mid', 'Senior'])

print("\n" + "="*60)
print("FEATURE ENGINEERING: NEW FEATURES CREATED")
print("="*60)
print(df_feat)

print("\n✅ Height_squared: Polynomial")
print("✅ Height_x_Weight: Interaction")
print("✅ BMI: Domain knowledge")
print("✅ Weight_per_cm: Ratio")
print("✅ Age_Group: Binning")

In [ ]:
# Example: Show how BMI feature helps prediction
# (BMI is a better predictor than raw height/weight)

print("\nWhy new features help:")
print("\nOriginal features: Height, Weight (2 independent variables)")
print("Model needs to learn: 'short heavy people are different from tall heavy people'")
print("\nWith BMI feature: Height, Weight, BMI (BMI already captures the relationship)")
print("Model can directly use: 'high BMI → higher risk of disease'")
print("\n💡 Good features make the model's job easier!")

## Feature Selection 🎯

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.datasets import make_classification

# Create dataset with many features (10 total)
X, y = make_classification(n_samples=200, n_features=10, n_informative=3,
                          n_redundant=3, n_repeated=2, random_state=42)

print("Dataset: 200 samples, 10 features")
print(f"Classes: {len(np.unique(y))}")
print(f"\nFeature breakdown:")
print(f"  - 3 informative features (important!)")
print(f"  - 3 redundant features (duplicates of informative)")
print(f"  - 2 repeated features (noise)")
print(f"  - 2 random features (pure noise)")

In [ ]:
# Method 1: SelectKBest with F-statistic
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X, y)

# Get scores
scores = selector.scores_
feature_names = [f'Feature {i}' for i in range(X.shape[1])]

# Sort by importance
indices = np.argsort(scores)[::-1]

print("\n" + "="*60)
print("METHOD 1: SelectKBest (F-statistic)")
print("="*60)
print(f"\nFeature Importance Scores:")
for i in range(len(scores)):
    idx = indices[i]
    star = "⭐" if i < 5 else " "
    print(f"{star} {feature_names[idx]:12s}: {scores[idx]:7.2f}")

print(f"\n✅ Top 5 features selected (marked with ⭐)")

In [ ]:
# Method 2: Mutual Information
selector_mi = SelectKBest(score_func=mutual_info_classif, k=5)
selector_mi.fit(X, y)

scores_mi = selector_mi.scores_
indices_mi = np.argsort(scores_mi)[::-1]

print("\n" + "="*60)
print("METHOD 2: Mutual Information")
print("="*60)
print(f"\nFeature Importance Scores:")
for i in range(len(scores_mi)):
    idx = indices_mi[i]
    star = "⭐" if i < 5 else " "
    print(f"{star} {feature_names[idx]:12s}: {scores_mi[idx]:7.3f}")

print(f"\n✅ Captures non-linear relationships")
print(f"✅ Better for complex patterns")

In [ ]:
# Method 3: Tree-based feature importance
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances_rf = rf.feature_importances_
indices_rf = np.argsort(importances_rf)[::-1]

print("\n" + "="*60)
print("METHOD 3: Random Forest Feature Importance")
print("="*60)
print(f"\nFeature Importance:")
for i in range(len(importances_rf)):
    idx = indices_rf[i]
    star = "⭐" if i < 5 else " "
    print(f"{star} {feature_names[idx]:12s}: {importances_rf[idx]:.4f}")

print(f"\n✅ Uses actual model predictions")
print(f"✅ Best for tree-based models")

In [ ]:
# Visualize feature importance comparison
plt.figure(figsize=(12, 6))

x_pos = np.arange(10)
width = 0.25

# Normalize scores for comparison
scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
scores_mi_norm = (scores_mi - scores_mi.min()) / (scores_mi.max() - scores_mi.min())
importances_rf_norm = importances_rf / importances_rf.max()

plt.bar(x_pos - width, scores_norm, width, label='F-statistic', alpha=0.8)
plt.bar(x_pos, scores_mi_norm, width, label='Mutual Information', alpha=0.8)
plt.bar(x_pos + width, importances_rf_norm, width, label='Random Forest', alpha=0.8)

plt.xlabel('Features', fontsize=12)
plt.ylabel('Importance (Normalized)', fontsize=12)
plt.title('Feature Selection Methods Comparison', fontsize=14)
plt.xticks(x_pos, [f'F{i}' for i in range(10)])
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Notice: Different methods rank features slightly differently!")
print("Combine multiple methods for robust feature selection.")

## Complete Feature Engineering Pipeline 🔄

In [ ]:
# Real-world example: Housing price prediction
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create realistic data
np.random.seed(42)
n_samples = 100

housing_data = {
    'Square_Feet': np.random.uniform(1000, 5000, n_samples),
    'Bedrooms': np.random.randint(1, 6, n_samples),
    'Bathrooms': np.random.uniform(1, 4, n_samples),
    'Year_Built': np.random.randint(1950, 2023, n_samples),
    'Price': np.random.uniform(200000, 1000000, n_samples)
}

df_housing = pd.DataFrame(housing_data)
print("Raw Housing Data:")
print(df_housing.head())
print(f"\nShape: {df_housing.shape}")

In [ ]:
# Step 1: Feature Engineering
df_eng = df_housing.copy()

# Create new features
df_eng['Price_per_sqft'] = df_eng['Price'] / df_eng['Square_Feet']
df_eng['Bath_per_Bed'] = df_eng['Bathrooms'] / df_eng['Bedrooms']
df_eng['House_Age'] = 2024 - df_eng['Year_Built']
df_eng['Total_Rooms'] = df_eng['Bedrooms'] + df_eng['Bathrooms']

print("\n" + "="*60)
print("STEP 1: FEATURE ENGINEERING")
print("="*60)
print(df_eng.head())
print(f"\nNew features created:")
print(f"  - Price_per_sqft (domain knowledge)")
print(f"  - Bath_per_Bed (ratio)")
print(f"  - House_Age (transformation)")
print(f"  - Total_Rooms (combination)")

In [ ]:
# Step 2: Handle missing data
# (Simulate some missing values)
df_missing = df_eng.copy()
missing_indices = np.random.choice(df_missing.index, 5, replace=False)
df_missing.loc[missing_indices, 'Bathrooms'] = np.nan

print("\n" + "="*60)
print("STEP 2: HANDLE MISSING DATA")
print("="*60)

imputer = SimpleImputer(strategy='median')
df_missing['Bathrooms'] = imputer.fit_transform(df_missing[['Bathrooms']])

print(f"Missing values: {df_missing.isnull().sum().sum()} (all filled)")
print(f"Strategy used: Median imputation")

In [ ]:
# Step 3: Feature Selection
X = df_missing.drop('Price', axis=1)
y = df_missing['Price']

print("\n" + "="*60)
print("STEP 3: FEATURE SELECTION")
print("="*60)

from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=5)
selector.fit(X, y)

selected_features = X.columns[selector.get_support()].tolist()
print(f"\nOriginal features: {len(X.columns)}")
print(f"Selected features: {len(selected_features)}")
print(f"\nSelected: {selected_features}")

X_selected = X[selected_features]

In [ ]:
# Step 4: Scaling
print("\n" + "="*60)
print("STEP 4: SCALING")
print("="*60)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

print(f"\nBefore scaling:")
print(f"  Min: {X_selected.min().min():.2f}")
print(f"  Max: {X_selected.max().max():.2f}")
print(f"  Mean: {X_selected.mean().mean():.2f}")

print(f"\nAfter scaling:")
print(f"  Min: {X_scaled.min():.2f}")
print(f"  Max: {X_scaled.max():.2f}")
print(f"  Mean: {X_scaled.mean():.2f}")

In [ ]:
# Step 5: Train model with engineered features
print("\n" + "="*60)
print("STEP 5: TRAIN MODEL")
print("="*60)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = r2_score(y_train, model.predict(X_train))
test_r2 = r2_score(y_test, model.predict(X_test))
train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))

print(f"\nTraining R²: {train_r2:.4f}")
print(f"Test R²:     {test_r2:.4f}")
print(f"\nTraining MAE: ${train_mae:,.0f}")
print(f"Test MAE:     ${test_mae:,.0f}")

print(f"\n✅ Model trained on engineered features!")

## Feature Engineering Best Practices 📋

In [ ]:
best_practices = """
🎯 FEATURE ENGINEERING BEST PRACTICES:

1️⃣  START WITH DOMAIN KNOWLEDGE
    - Understand the problem deeply
    - Talk to domain experts
    - Feature ideas often come from domain, not data

2️⃣  EXPLORATORY DATA ANALYSIS (EDA)
    - Plot distributions
    - Check correlations
    - Identify outliers and missing data
    - Understand data relationships

3️⃣  HANDLE MISSING DATA FIRST
    - Don't ignore missing values!
    - Choose method based on % missing:
      < 5% → Delete rows
      5-20% → Impute (mean/median/KNN)
      > 20% → Create indicator feature

4️⃣  CREATE MEANINGFUL FEATURES
    - Domain-specific (BMI from height/weight)
    - Ratios and differences
    - Polynomial features (for ML, not linear)
    - Interactions (when features combine)
    - Binning/discretization (for tree models)

5️⃣  SCALE/NORMALIZE APPROPRIATELY
    - Standardize for: Neural networks, SVM, KNN
    - Skip for: Trees, forests, boosting
    - Always fit on train, apply to test

6️⃣  ENCODE CATEGORICAL VARIABLES
    - Label encode: Ordinal categories
    - One-hot encode: Nominal categories (< 10 unique)
    - Target encode: High-cardinality (be careful!)

7️⃣  SELECT FEATURES STRATEGICALLY
    - Don't use all features!
    - Remove: Constant features, duplicates, highly correlated
    - Use: SelectKBest, tree importance, correlation analysis
    - More features = longer training, worse generalization

8️⃣  AVOID DATA LEAKAGE
    - ❌ DON'T: Fit scaler on ALL data, then split
    - ✅ DO: Split first, fit scaler on train, apply to test
    - ❌ DON'T: Include information only available at prediction time
    - ✅ DO: Only use features available for real predictions

9️⃣  VALIDATE FEATURES
    - Do new features improve validation score?
    - Can you interpret what they mean?
    - Are they stable across different data splits?

🔟  ITERATE!
    - Feature engineering is iterative
    - Start simple, add complexity as needed
    - Spend 80% of time here, 20% on algorithms
"""

print(best_practices)

## Key Takeaways 🎯

✅ Feature engineering is often more important than algorithm choice

✅ Handle missing data appropriately (delete, impute, or indicate)

✅ Scale numerical features for distance-based algorithms

✅ Encode categorical variables (label or one-hot depending on type)

✅ Create domain-meaningful features (ratios, combinations, interactions)

✅ Select most important features to reduce dimensionality

✅ Always validate improvements on test data

✅ Avoid data leakage (split before scaling/imputation)

## Practice Exercise 💪

1. **Create a messy dataset:** Add missing values, outliers, mixed scales
2. **Clean it:** Handle missing data, scale, encode categories
3. **Engineer features:** Create 3-5 meaningful new features
4. **Compare:** Train model with original features vs engineered
5. **Select:** Use SelectKBest to find top features
6. **Evaluate:** Check if engineering actually helped!

Challenge: Improve a model's test accuracy by 5% using ONLY feature engineering!

## Next Steps 🚀

→ **Model Validation:** Cross-validation, hyperparameter tuning

→ **Imbalanced Data:** Handling skewed class distributions

→ **Projects:** Apply to real Kaggle competitions!

→ **Advanced:** Automated feature engineering (AutoML)